# Lesson 3: Artefact correction

Neurocampus course "Signals of the whole brain"

Daria Kleeva

dkleeva@gmail.com

March 11, 2026


## Look at the data we recorded

<div style="color: blue;">

Let's look at the data we collected at the previous lesson. Import the file 'probe.edf', explore its properties, look at time series. What patterns do you notice? 

Filter the data, set up the correct montage. Plot PSD.

</div>

In [ ]:
#...

## Annotations

In [ ]:
%matplotlib qt
raw.plot()


In [ ]:
import numpy as np

intervals = [
    (0.0, 50.0, "vertical EOG"),
    (52.0, 68.0, "horizontal EOG"),
    (82.0, 89.0, "EMG artefacts"),
    (105.0, 110.0, "Cough artefacts"),
    (216.0, 288.0, "Eyes open"),
    (314.0, 396.0, "Eyes closed")
]

onsets = [start for start, end, lbl in intervals]
durations = [end - start for start, end, lbl in intervals]
descriptions = [lbl for start, end, lbl in intervals]

ann = mne.Annotations(
    onset=onsets,
    duration=durations,
    description=descriptions,
)

# Replace existing annotations:
raw.set_annotations(ann)

# Or append to existing:
# raw.set_annotations(raw.annotations + ann)

print(raw.annotations)

In [ ]:
raw.plot()

## Spectral characteristics of various segments

In [ ]:
from collections import defaultdict


segments = defaultdict(list)

for ann in raw.annotations:
    desc = ann["description"]
    tmin = ann["onset"]
    tmax = ann["onset"] + ann["duration"]

    seg = raw.copy().crop(tmin=tmin, tmax=tmax, include_tmax=False)
    segments[desc].append(seg)

raws = {}
for desc, seg_list in segments.items():
    if len(seg_list) == 1:
        raws[desc] = seg_list[0]
    else:
        raws[desc] = mne.concatenate_raws(seg_list)

print(raws.keys())

In [ ]:
raws

In [ ]:
raws['EMG artefacts'].plot_psd(fmax=40)

In [ ]:
raw.info['ch_names']

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

regions = {
    "Frontal":   ['Fp1','Fpz','Fp2','F7','F3','Fz','F4','F8'],
    "Central":   ["C3", "C4", "Cz"],
    "Parietal":  ["P3", "P4", "Pz", "P5", "P6"],
    "Temporal":  ["T3", "T4", "TP7", "TP8"],
    "Occipital": ["O1", "O2", "Oz"],
}


fmin, fmax = 1, 40
n_fft = 2048

n_regions = len(regions)
n_cols = 3
n_rows = int(np.ceil(n_regions / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 3.8 * n_rows), sharex=True, sharey=True)
axes = np.atleast_1d(axes).ravel()

condition_styles = {
    "Eyes open": {"color": "tab:blue"},
    "Eyes closed": {"color": "tab:orange"},
}

for ax, (region_name, ch_list) in zip(axes, regions.items()):
    any_plotted = False

    for cond_name, raw in raws.items():
        if cond_name in ['Eyes open', 'Eyes closed']:
            available = [ch for ch in ch_list if ch in raw.ch_names]
            if len(available) == 0:
                continue
            spectrum = raw.compute_psd(
                method="welch",
                picks=available,
                fmin=fmin,
                fmax=fmax,
                n_fft=n_fft,
                reject_by_annotation=True,
                verbose=False,
            )
            psds, freqs = spectrum.get_data(return_freqs=True)

            psds_db = 10 * np.log10(psds)
            mean_db = psds_db.mean(axis=0)
            std_db = psds_db.std(axis=0)

            style = condition_styles.get(cond_name, {})
            ax.plot(freqs, mean_db, label=cond_name, linewidth=2, **style)
            ax.fill_between(freqs, mean_db - std_db, mean_db + std_db, alpha=0.2, **style)
            any_plotted = True

    ax.set_title(region_name)
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("PSD (dB/Hz)")
    ax.grid(alpha=0.3)
    if any_plotted:
        ax.legend(frameon=False)

for i in range(len(regions), len(axes)):
    axes[i].axis("off")

fig.suptitle("Regional PSD comparison: eyes_closed vs eyes_open", y=1.02, fontsize=14)
fig.tight_layout()
plt.show()

<div style="color: blue;">
Change EEG reference to average and look at the spectra again.
</div>

In [ ]:
#...

## Handling the bad channels

We recorded clear data, so let's create the noisy channel manually

In [ ]:
import numpy as np

raw_ec = raws['Eyes closed']

f8_idx = raw_ec.ch_names.index('F8')

data = raw_ec.get_data()

# Manually add Gaussian noise to the F8 channel
rng = np.random.default_rng(42)
noise_std = 20e-6  
data[f8_idx] += rng.normal(0, noise_std, size=data.shape[1])

raw_ec._data = data

In [ ]:
%matplotlib qt
raws['Eyes closed'].plot()

In [ ]:
raws['Eyes closed'].info['bads']=['F8']

**Interpolation** of bad channels means replacing the signal in channels marked as bad with an estimate derived from neighboring 'good' sensors. For EEG, MNE typically uses spherical-spline interpolation (a smooth spatial model over the scalp), while for MEG it uses sensor-field interpolation based on the measured field structure. 

In [ ]:
raws['Eyes closed'].interpolate_bads(reset_bads=True)

In [ ]:
raws['Eyes closed'].plot()

## Repairing artefacts with SSP

**Signal Space Projection (SSP)** repairs artifacts by identifying spatial patterns (across sensors) that correspond to unwanted activity (most often eye blinks, ECG, or environmental noise), and then projecting the data onto the subspace orthogonal to those patterns, effectively removing their contribution. Practically, you compute projection vectors from data segments dominated by an artifact, inspect and select how many components to keep as projectors, and apply them to raw/epochs/evoked data; this is linear, fast, and preserves most neural signal when component selection is careful. 

In [ ]:
from mne.preprocessing import (
    compute_proj_eog,
    create_eog_epochs,
)

In [ ]:
%matplotlib inline
eog = create_eog_epochs(raws['vertical EOG'],ch_name='Fp1').average()
eog.plot_joint()
plt.show()

In [ ]:
projs, events = compute_proj_eog(raws['vertical EOG'], ch_name='Fp1', n_eeg=1, reject=None)

In [ ]:
mne.viz.plot_projs_topomap(projs, info=raw.info)
plt.show()

In [ ]:
raws['vertical EOG'].add_proj(projs)

In [ ]:
%matplotlib qt
raws['vertical EOG'].plot()
#shift+j to toggle projectors

In [ ]:
%matplotlib inline
mne.viz.plot_projs_joint(projs, eog, picks_trace="Fp1")
plt.show()

## Independent component analysis

**Independent Component Analysis (ICA)** is a blind source separation method that assumes observed multichannel signals are linear mixtures of statistically independent latent sources. Formally, we observe $ X = AS $, where $ S $ contains unknown independent components and $ A $ is an unknown mixing matrix; ICA estimates an unmixing matrix $ W \approx A^{-1} $ such that $ \hat{S} = WX $ maximizes statistical independence between components. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import FastICA
from scipy import signal

Let's create independent sources

In [ ]:
rng = np.random.default_rng(0)
sfreq = 250  # Hz
T = 6.0      # seconds
t = np.arange(int(T * sfreq)) / sfreq
n = t.size

s1 = np.sin(2 * np.pi * 8 * t)                                  # sinusoid (8 Hz)
s2 = signal.sawtooth(2 * np.pi * 2 * t, width=0.3)              # non-sinusoidal wave (2 Hz)
s3 = rng.laplace(loc=0.0, scale=1.0, size=n)        


fig, axs = plt.subplots(3, 1, figsize=(10, 6), sharex=True)

axs[0].plot(t, s1, label='Sinusoid (8 Hz)')
axs[0].set_ylabel('Amplitude')
axs[0].legend(loc='upper right')

axs[1].plot(t, s2, label='Sawtooth (2 Hz)', color='orange')
axs[1].set_ylabel('Amplitude')
axs[1].legend(loc='upper right')

axs[2].plot(t, s3, label='Laplace (0 Hz)', color='green')
axs[2].set_ylabel('Amplitude')
axs[2].set_xlabel('Time (s)')
axs[2].legend(loc='upper right')

plt.tight_layout()

Are the sources Gaussian?

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(10, 6), sharex=True)

axs[0].hist(s1, bins=20)
axs[0].set_ylabel('Frequency')
axs[0].set_title('Sinusoid (8 Hz)')

axs[1].hist(s2, bins=20)
axs[1].set_ylabel('Frequency')
axs[1].set_title('Sawtooth (2 Hz)')

axs[2].hist(s3, bins=20)
axs[2].set_ylabel('Frequency')
axs[2].set_title('Laplace (0 Hz)')

plt.tight_layout()


from scipy.stats import kurtosis
print(kurtosis(s1))
print(kurtosis(s2))
print(kurtosis(s3))




It is OK to have at most one Gaussian source for ICA. Then, the decomposition is still identifiable. 

Why:
- Gaussian distributions are rotationally symmetric.
- Any orthogonal rotation of Gaussian variables is still Gaussian.
- If we had more than one Gaussian sources, we could rotate them arbitrarily and ICA would have no unique solution.

In [ ]:
S = np.c_[s1, s2, s3]
S = (S - S.mean(axis=0)) / S.std(axis=0)

Now we mix the sources (in real life we do not know the mixing matrix)

In [ ]:
A = np.array([[1.0,  0.5, 0.2],
              [0.3,  1.0, 0.4],
              [0.2, -0.2, 1.0]])  

X = S @ A.T 

#Add some noise to look realistic
X += 0.02 * rng.standard_normal(size=X.shape)

fig, axs = plt.subplots(3, 1, figsize=(10, 6), sharex=True)

axs[0].plot(t, X[:, 0], label='Mixed signal 1')
axs[0].set_ylabel('Amplitude')
axs[0].legend(loc='upper right')

axs[1].plot(t, X[:, 1], label='Mixed signal 2')
axs[1].set_ylabel('Amplitude')
axs[1].legend(loc='upper right')    

axs[2].plot(t, X[:, 2], label='Mixed signal 3')
axs[2].set_ylabel('Amplitude')
axs[2].set_xlabel('Time (s)')
axs[2].legend(loc='upper right')

plt.tight_layout()



Run ICA on mixtures

In [ ]:
ica = FastICA(n_components=3, whiten="unit-variance", random_state=0, max_iter=2000)
S_hat = ica.fit_transform(X)     
A_hat = ica.mixing_   

In [ ]:
A_hat

In [ ]:
A

Let's look how well we recovered the signal

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(10, 6), sharex=True)

axs[0].plot(t, S_hat[:, 0], label='Reconstructed source 1')
axs[0].set_ylabel('Amplitude')
axs[0].legend(loc='upper right')

axs[1].plot(t, S_hat[:, 1], label='Reconstructed source 2')
axs[1].set_ylabel('Amplitude')
axs[1].legend(loc='upper right')
plt.plot(t, S_hat[:, 2], label='Reconstructed source 3')
plt.legend()
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.tight_layout()

Now let's switch to the real data

In [ ]:
raw_for_ica = raws['vertical EOG'].copy()
raw_for_ica.del_proj()

In [ ]:
raw_for_ica.plot()

In [ ]:
from mne.preprocessing import ICA

In [ ]:
ica = ICA(n_components=15, max_iter="auto", random_state=97)
ica.fit(raw_for_ica)

In [ ]:
explained_var_ratio = ica.get_explained_variance_ratio(raw_for_ica)
print(f"Fraction of variance explained by all components: {explained_var_ratio}")


In [ ]:
%matplotlib qt
ica.plot_sources(raw_for_ica)

In [ ]:
%matplotlib inline
ica.plot_components()
plt.show()

In [ ]:
fig = ica.plot_overlay(raw_for_ica.copy().crop(40,45), exclude=[0], picks="eeg")

In [ ]:
ica.plot_properties(raw_for_ica, picks=range(15))

In [ ]:
ica.exclude = []
clean_raw = raw_for_ica.copy()
ica.apply(clean_raw)


In [ ]:
%matplotlib qt
clean_raw.plot()

<div style="color: blue;">

Try:
- Different number of components;
- Different segments;
- Fit ICA on one segment, apply to another;
- Different ICA algorithm

</div>